# Making Roman Mosaics

## Introduction

This notebook demonstrates the use of ROSALIA to make mosaics in MAST on commissioning data products.

## Imports

In [1]:
import rosalia as rs
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord  # High-level coordinates

In [2]:
rs.utils.save_dict

<function rosalia.utils.save_dict(dictionary, input_name, verbose=False)>

ROSALIA assumes you have a valid MAST AUTH token to access the requested data.  

If you do not have a MAST AUTH token for accessing MAST (needed for Roman OPS access until after commissioning), or if your token has expired, make one now by visiting [MAST.Auth](https://auth.mast.stsci.edu/info).

Once you have a token, you can set it up as an environment variable for later use, or you can store it in a file called `variables.env`. Below, we have assumed the file is in your current working directory (`./`), but the full path to the file can be specified if it is stored elsewhere such as your home directory. The `variables.env` file should look like:

```
MAST_API_TOKEN='yourtokenhere'
```

Here we assume you stored it as an environment variable (recommended). The line to add to your .bashrc/.zshrc would look like:

```
export MAST_API_TOKEN='yourtokenhere'
```

In [3]:
import os
from astroquery.mast import MastMissions
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path="./variables.env")

# Create MastMissions object and assign mission to 'roman'
missions = MastMissions(mission='roman')

# Login to search and retrieve Roman data
token = os.getenv("MAST_API_TOKEN")    
missions.login(token=token)
               
print(f'Mission: {missions.mission}')
print(f'Service: {missions.service}')

INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]
Mission: roman
Service: search


In [4]:
"""
Example of criteria for a roman_query using astroquery.mast.query_criteria
search = {'program': 1020,
         'observation': 5,
         'pass': 2, 
         #'exposure_start_time': '2026-09-15T02:01:47.4080000',
         'product_type': 'l2', 
         'detector': 'WFI06',
         # 'optical_element': 'F158',
         'exposure_type': "WFI_IMAGE"}

Here we use a region criteria, based on a set of coordinates and a search radius.


    program_id = visit_id[0:5]
    plan_id = visit_id[5:7]
    pass_id = visit_id[7:10]
    segment_id = visit_id[10:13]

    'exposure_id': '0102203058001001001',
01022 03 058 001 001 001
"""

#coordinates=SkyCoord(229.64044*u.deg, -3.58747*u.deg, frame="icrs")
#radius=10*u.arcsec

# query = roman_query_criteria(search=search, file_suffix="_cal")
# results, products = rs.mast.roman_query(coordinates=coordinates, radius=radius, file_suffix="_cal")


search = {'program': 1022,
          #'execution_plan': 3,
          #'pass': 53,
          #'segment': 1,
          # 'observation': 2,
          # 'visit': 1,
          'product_type': 'l2', 
          'exposure_type': "WFI_IMAGE",
          }

results, products, exposures = rs.mast.roman_query(search, file_suffix="_cal")

INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]
Mission: roman
Service: search
Fetching products for 1832 unique datasets in 2 batches ... [Failed]


KeyboardInterrupt: 

In [ ]:
obsid = []
exptime = []
sca = []
mjd = []
ra_target = []
dec_target = [] 
pav3 = [] 
pa_aper  = []
obspos = []
bandpass = []
median_back = []
rms_back = []
good_pixel_fraction = [] 

from tqdm import tqdm  
# for i in tqdm(range(100)):
for i in tqdm(range(len(products))):
    stream = rs.mast.stream_roman_mast(products, row=i)
    obsid.append(stream["meta"]["filename"])
    exptime.append(stream["meta"]["exposure"]["exposure_time"])
    sca.append(stream["meta"]["instrument"]["detector"])
    mjd.append(stream["meta"]["ephemeris"]['time'])
    ra_target.append(stream["meta"]["pointing"]["target_ra"])
    dec_target.append(stream["meta"]["pointing"]["target_dec"])
    pav3.append(stream["meta"]["pointing"]["pa_v3"])
    pa_aper.append(stream["meta"]["pointing"]["pa_aperture"])
    obspos.append([stream["meta"]["ephemeris"]['spatial_x'], stream["meta"]["ephemeris"]['spatial_y'], stream["meta"]["ephemeris"]['spatial_z']])
    bandpass.append(stream["meta"]["instrument"]["optical_element"])
    median_back.append(stream["meta"]["statistics"]["image_median"])
    rms_back.append(stream["meta"]["statistics"]["image_rms"])
    good_pixel_fraction.append(stream["meta"]["statistics"]["good_pixel_fraction"])

CAR170_db = {"obsid": obsid, "exptime": exptime, "sca": sca, "mjd": mjd, "ra_target": ra_target,
             "dec_target": dec_target, "pav3": pav3, "pa_aper": pa_aper, "obspos": obspos, "bandpass": bandpass,
             "median_back": median_back, "rms_back": rms_back, "good_pixel_fraction": good_pixel_fraction}

rs.utils.save_dict(CAR170_db, "CAR170_db.pkl") 

In [ ]:
# Get the planets 

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(ra_target, dec_target)

In [ ]:
plt.scatter(ra_target, median_back)

In [ ]:
stream["meta"]["pointing"]

In [ ]:
stream["meta"]["exposure"]

In [ ]:
stream["meta"].keys()

In [ ]:
results

In [ ]:
roman_exposure = rs.core.exposure(products)

In [ ]:
roman_exposure.save_drz(outname=exposures[0] + ".fits", resolution=0.11)
roman_exposure.straylight()


In [ ]:
roman_exposure.SCIEXTS

In [ ]:
roman_exposure.make_close_stars_ds9_region()

## About this Notebook

**Author:** Alejandro S. Borlaff - NASA Ames Research Center (Code S) - a.s.borlaff@nasa.gov 

**Updated on** September 20, 2026. 